# Linear regression model


## Answer the questions

### Derive an analytical solution to the regression problem. Use a vector form of the equation.


![Решение аналитически](data/linreg_solution_vector.jpg)

### What changes in the solution when L1 and L2 regularizations are added to the loss function.


![Решение аналитически](data/logreg_l1_l2.jpg)

### Explain why L1 regularization is often used to select features. Why are there many weights equal to 0 after the model is fit?

![Решение аналитически](data/l1_l2_plots.jpg)

веса подбираются так, чтобы они были внутри границы регуляризации, но при этом решение было максимально точным
точка на графике - идеальное решение, стремясь быть максимально близко к точному решению и при этом остаться внутри границы, в L1 мы попадаем на ось, в которой вес зануляется, в отличие от L2

с точки зрения градиента:

в L2 штраф это производная от w**2 -- 2w. то есть он зависит от самого значения веса

в L1 же штраф это производная от |w| -- 1, -1. то есть он не зависит от значения веса и может продавить вес до значения нуля

### Explain how you can use the same models (Linear regression, Ridge, etc.) but make it possible to fit nonlinear dependencies.

в уравнение регрессии y = wx + b подставить x, например, со степенью, тогда у нас получится y = w1*x1 + w2*x2**2 + b

тогда у нас получится полиномиальная регрессия и увидим на графике параболу

## Introduction — make all the preprocessing staff from the previous lesson

### Import libraries.


In [98]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import lightgbm
import scipy
import statsmodels
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

In [99]:
train = pd.read_json('data/train.json')
low = train['price'].quantile(0.01)
high = train['price'].quantile(0.99)
train = train.copy()[(train['price'] > low) & (train['price'] < high)].copy()
train

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, Fitness Center, Laundry in...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124000,1.0,3,92bbbf38baadfde0576fc496bd41749c,2016-04-05 03:58:33,There is 700 square feet of recently renovated...,W 171 Street,"[Elevator, Dishwasher, Hardwood Floors]",40.8433,6824800,-73.9396,a61e21da3ba18c7a3d54cfdcc247e1f8,[https://photos.renthop.com/2/6824800_0682be16...,2800,620 W 171 Street,low
124002,1.0,2,5565db9b7cba3603834c4aa6f2950960,2016-04-02 02:25:31,"2 bedroom apartment with updated kitchen, rece...",Broadway,"[Common Outdoor Space, Cats Allowed, Dogs Allo...",40.8198,6813268,-73.9578,8f90e5e10e8a2d7cf997f016d89230eb,[https://photos.renthop.com/2/6813268_1e6fcc32...,2395,3333 Broadway,medium
124004,1.0,1,67997a128056ee1ed7d046bbb856e3c7,2016-04-26 05:42:03,No Brokers Fee * Never Lived 1 Bedroom 1 Bathr...,210 Brighton 15th St,"[Dining Room, Elevator, Pre-War, Laundry in Bu...",40.5765,6927093,-73.9554,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/6927093_93a52104...,1850,210 Brighton 15th St,medium
124008,1.0,2,3c0574a740154806c18bdf1fddd3d966,2016-04-19 02:47:33,Wonderful Bright Chelsea 2 Bedroom apartment o...,West 21st Street,"[Pre-War, Laundry in Unit, Dishwasher, No Fee,...",40.7448,6892816,-74.0017,c3cd45f4381ac371507090e9ffabea80,[https://photos.renthop.com/2/6892816_1a8d087a...,4195,350 West 21st Street,medium


## Intro data analysis part 2



### Let's generate additional features for better model quality. Consider a column called "Features". It consists of a list of highlights of the current flat.

In [100]:
train['features']

4         [Dining Room, Pre-War, Laundry in Building, Di...
6         [Doorman, Elevator, Laundry in Building, Dishw...
9         [Doorman, Elevator, Laundry in Building, Laund...
10                                                       []
15        [Doorman, Elevator, Fitness Center, Laundry in...
                                ...                        
124000              [Elevator, Dishwasher, Hardwood Floors]
124002    [Common Outdoor Space, Cats Allowed, Dogs Allo...
124004    [Dining Room, Elevator, Pre-War, Laundry in Bu...
124008    [Pre-War, Laundry in Unit, Dishwasher, No Fee,...
124009    [Dining Room, Elevator, Laundry in Building, D...
Name: features, Length: 48343, dtype: object

### Remove unused symbols ([,], ', ", and space) from the column.

In [101]:
train['features'] = (train['features'].astype(str).str.replace('[', '', regex=False).str.replace(']', '', regex=False).str.replace("'", "", regex=False).str.replace('"', '', regex=False).str.replace(' ', '', regex=False))
train['features']

4         DiningRoom,Pre-War,LaundryinBuilding,Dishwashe...
6         Doorman,Elevator,LaundryinBuilding,Dishwasher,...
9         Doorman,Elevator,LaundryinBuilding,LaundryinUn...
10                                                         
15         Doorman,Elevator,FitnessCenter,LaundryinBuilding
                                ...                        
124000                   Elevator,Dishwasher,HardwoodFloors
124002    CommonOutdoorSpace,CatsAllowed,DogsAllowed,NoF...
124004    DiningRoom,Elevator,Pre-War,LaundryinBuilding,...
124008    Pre-War,LaundryinUnit,Dishwasher,NoFee,Outdoor...
124009    DiningRoom,Elevator,LaundryinBuilding,Dishwash...
Name: features, Length: 48343, dtype: object

### Get all values in each list and collect the result in one huge list for the whole dataset. You can use DataFrame.iterrows().

In [102]:
# big_list = []
# for i, row in train.iterrows():
#     big_list.extend(row['features'].split(','))
# big_list

In [103]:
big_list = []
for i, row in train.iterrows():
    for i in row['features'].split(','):
        if i.strip():
            big_list.append(i.strip())
big_list

['DiningRoom',
 'Pre-War',
 'LaundryinBuilding',
 'Dishwasher',
 'HardwoodFloors',
 'DogsAllowed',
 'CatsAllowed',
 'Doorman',
 'Elevator',
 'LaundryinBuilding',
 'Dishwasher',
 'HardwoodFloors',
 'NoFee',
 'Doorman',
 'Elevator',
 'LaundryinBuilding',
 'LaundryinUnit',
 'Dishwasher',
 'HardwoodFloors',
 'Doorman',
 'Elevator',
 'FitnessCenter',
 'LaundryinBuilding',
 'Doorman',
 'Elevator',
 'Loft',
 'Dishwasher',
 'HardwoodFloors',
 'NoFee',
 'Fireplace',
 'LaundryinUnit',
 'Dishwasher',
 'HardwoodFloors',
 'NoFee',
 'Elevator',
 'LaundryinBuilding',
 'Dishwasher',
 'HardwoodFloors',
 'NoFee',
 'HardwoodFloors',
 'CatsAllowed',
 'DogsAllowed',
 'Doorman',
 'Elevator',
 'LaundryinBuilding',
 'DogsAllowed',
 'CatsAllowed',
 'RoofDeck',
 'Doorman',
 'Elevator',
 'FitnessCenter',
 'Pre-War',
 'LaundryinBuilding',
 'HighSpeedInternet',
 'Dishwasher',
 'HardwoodFloors',
 'NoFee',
 'DogsAllowed',
 'CatsAllowed',
 'SwimmingPool',
 'RoofDeck',
 'Doorman',
 'Elevator',
 'FitnessCenter',
 'Laun

### Count the most popular functions from our huge list and take the top 20 for this moment.

In [104]:
counts = Counter(big_list)
most_common_features = counts.most_common(20)
most_common_features

[('Elevator', 25375),
 ('HardwoodFloors', 23146),
 ('CatsAllowed', 23135),
 ('DogsAllowed', 21652),
 ('Doorman', 20479),
 ('Dishwasher', 20081),
 ('NoFee', 17793),
 ('LaundryinBuilding', 16082),
 ('FitnessCenter', 12989),
 ('Pre-War', 8971),
 ('LaundryinUnit', 8437),
 ('RoofDeck', 6417),
 ('OutdoorSpace', 5132),
 ('DiningRoom', 4890),
 ('HighSpeedInternet', 4223),
 ('Balcony', 2898),
 ('SwimmingPool', 2643),
 ('LaundryInBuilding', 2564),
 ('NewConstruction', 2504),
 ('Terrace', 2177)]

In [105]:
features = []
for i in most_common_features:
    features.append(i[0])
features_series = pd.Series(features, name = 'features')
features_series

0              Elevator
1        HardwoodFloors
2           CatsAllowed
3           DogsAllowed
4               Doorman
5            Dishwasher
6                 NoFee
7     LaundryinBuilding
8         FitnessCenter
9               Pre-War
10        LaundryinUnit
11             RoofDeck
12         OutdoorSpace
13           DiningRoom
14    HighSpeedInternet
15              Balcony
16         SwimmingPool
17    LaundryInBuilding
18      NewConstruction
19              Terrace
Name: features, dtype: object

In [106]:
y_train = train['price']
train = train[['bathrooms', 'bedrooms', 'features']]
train
mega_df = train['features'].str.get_dummies(sep=',')[features]
mega_df

,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
4,0,1,1,1,0,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0
6,1,1,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0
9,1,1,0,0,1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0
10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15,1,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124000,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
124002,1,0,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0
124004,1,1,1,1,0,1,1,1,0,1,1,0,0,1,0,0,0,0,0,0
124008,0,0,0,0,0,1,1,0,0,1,1,0,1,0,0,0,0,0,0,0


In [107]:
train = train.drop(columns='features')
X_train = pd.concat([train, mega_df], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=21)
X_train

,bathrooms,bedrooms,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,...,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
110878,1.0,1,1,1,1,1,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
67866,1.0,1,0,0,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
99860,1.0,2,1,0,1,1,1,0,0,0,...,0,0,1,0,0,0,0,0,0,1
9759,2.0,2,1,1,0,0,1,1,1,1,...,0,1,0,1,0,0,1,0,0,1
49188,1.0,2,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42246,1.0,3,1,1,0,0,1,0,1,1,...,1,0,0,0,0,0,0,0,0,0
23135,2.5,4,1,1,0,0,0,1,0,1,...,1,0,0,0,0,0,0,0,0,0
15367,1.0,2,1,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
13753,1.0,1,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 4. Models implementation — Linear regression

### Implement a Python class for a linear regression algorithm with two basic methods — fit and predict. Use stochastic gradient descent (SGD) to find optimal model weights. For better understanding, we recommend implementing separate versions of the algorithm with the analytical solution and non-stochastic gradient descent under the hood.


In [108]:
class SGDLinearRegression:
    def __init__(self, epoch, learning_rate=0.01, l1=0, l2=0):
        self.bias = 0
        self.epochs = epoch
        self.learning_rate = learning_rate
        self.l1 = l1
        self.l2 = l2
        self.batch_size = 64

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        self.weights = np.zeros(X.shape[1])
        for epoch in range(self.epochs):
            indexes = np.random.permutation(len(X))
            for i in range(0, len(indexes), self.batch_size):
                xi = X[i : i + self.batch_size]
                yi = y[i: i + self.batch_size]
                prediction = self.bias + (xi @ self.weights)
                errors = yi - prediction
                grad = (xi.T @ (- errors) / len(xi) + (2 * self.l2 * self.weights) / len(X)) + (self.l1 * np.sign(self.weights) / len(X))
                self.weights = self.weights - self.learning_rate * grad
                self.bias = self.bias - self.learning_rate * np.mean(-errors)

    def predict(self, X):
        X = np.array(X)
        return self.bias + (X @ self.weights)


In [109]:
class AnalLinearRegression:

    def __init__(self, l2 = 0):
        self.l2 = l2

    def fit(self, X, Y):
        x = np.array(X)
        y = np.array(Y)
        x = np.hstack([np.ones((x.shape[0], 1)), x])
        I = np.eye(x.shape[1])
        I[0,0] = 0
        self.weights = (np.linalg.inv(x.T @ x + self.l2 * I) @ x.T @ y)  # for matrix formula budet:  (Xt * X) ** -1 * Xt * y

    def predict(self, X):
        x = np.array(X)
        x = np.hstack([np.ones((x.shape[0], 1)), x])
        return x @ self.weights

In [110]:
class GradLinearRegression:
    def __init__(self, epoch, learning_rate=0.01, l1=0, l2=0):
        self.bias = 0
        self.epochs = epoch
        self.learning_rate = learning_rate
        self.l1 = l1
        self.l2 = l2

    def fit(self, X, Y):
        X = np.array(X)
        y = np.array(Y)
        self.weights = np.zeros(X.shape[1])
        for epoch in range(self.epochs):
            predict = (X @ self.weights) + self.bias
            grad = -(2 / len(X)) * (X.T @ (y - predict)) + (2 * self.l2 * self.weights) + (self.l1 * np.sign(self.weights))
            self.weights = self.weights - self.learning_rate * grad
            self.bias = self.bias - self.learning_rate * ( -(2 / len(X)) * np.sum(y - predict))

    def predict(self, X):
        return X @ self.weights + self.bias


In [111]:
qwe = GradLinearRegression(10000)
qwe.fit(X_train, y_train)
qwe.predict(X_train)

110878    2688.376089
67866     3317.437359
99860     4082.780452
9759      5542.943508
49188     3307.624704
             ...     
42246     4552.158764
23135     7064.285610
15367     3760.396652
13753     2691.365712
39273     2247.792935
Length: 38674, dtype: float64

### Define the R squared (R2) coefficient and implement a function to calculate it.

#### R2 это нормированная метрика которая определяется как 1 - MSE / дисперсия

In [112]:
def RSquared(model, X, Y):
    X = np.array(X)
    y = np.array(Y)
    predict = model.predict(X)
    MSE = np.sum((y - predict) ** 2)
    naive = np.sum((y - np.mean(y)) ** 2)
    return 1 - MSE/naive

### Make predictions with your algorithm and estimate the model with MAE, RMSE and R2 metrics.


#### Store the metrics as in the previous lesson in a table with columns model, train, test for MAE table, RMSE table, and R2 coefficient.

In [113]:
df_mae = pd.DataFrame(columns=['model', 'train', 'test'])
df_rmse = pd.DataFrame(columns=['model', 'train', 'test'])
df_r2 = pd.DataFrame(columns=['model', 'train', 'test'])

In [114]:
sgd_linreg = SGDLinearRegression(1000)
sgd_linreg.fit(X_train, y_train)

In [115]:
sgd_linreg_pred_train = sgd_linreg.predict(X_train)
sgd_linreg_pred_test = sgd_linreg.predict(X_test)

In [116]:
MAE_train = mean_absolute_error(y_train, sgd_linreg_pred_train)
MAE_test = mean_absolute_error(y_test, sgd_linreg_pred_test)
RMSE_train = np.sqrt(mean_squared_error(y_train, sgd_linreg_pred_train))
RMSE_test = np.sqrt(mean_squared_error(y_test, sgd_linreg_pred_test))
R2_train = RSquared(sgd_linreg, X_train, y_train)
R2_test = RSquared(sgd_linreg, X_test, y_test)

In [117]:
df_mae.loc[0] = ['SGDLinearRegression', MAE_train, MAE_test]
df_rmse.loc[0] = ['SGDLinearRegression', RMSE_train, RMSE_test]
df_r2.loc[0] = ['SGDLinearRegression', R2_train, R2_test]

In [118]:
linreg = LinearRegression()
linreg.fit(X_train, y_train)
pred_sklearn_train = linreg.predict(X_train)
pred_sklearn_test = linreg.predict(X_test)

In [119]:
MAE_train = mean_absolute_error(y_train, pred_sklearn_train)
MAE_test = mean_absolute_error(y_test, pred_sklearn_test)
RMSE_train = np.sqrt(mean_squared_error(y_train, pred_sklearn_train))
RMSE_test = np.sqrt(mean_squared_error(y_test, pred_sklearn_test))
R2_train = RSquared(linreg, X_train, y_train)
R2_test = RSquared(linreg, X_test, y_test)
MAE_train

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


708.4949982592885

In [120]:
df_mae.loc[1] = ['sklearnLinearRegression', MAE_train, MAE_test]
df_rmse.loc[1] = ['sklearnLinearRegression', RMSE_train, RMSE_test]
df_r2.loc[1] = ['sklearnLinearRegression', R2_train, R2_test]

In [121]:
analytic_linreg = AnalLinearRegression()
analytic_linreg.fit(X_train, y_train)
analytic_linreg_pred_train = analytic_linreg.predict(X_train)
analytic_linreg_pred_test = analytic_linreg.predict(X_test)

In [122]:
MAE_train = mean_absolute_error(y_train, analytic_linreg_pred_train)
MAE_test = mean_absolute_error(y_test, analytic_linreg_pred_test)
RMSE_train = np.sqrt(mean_squared_error(y_train, analytic_linreg_pred_train))
RMSE_test = np.sqrt(mean_squared_error(y_test, analytic_linreg_pred_test))
R2_train = RSquared(analytic_linreg, X_train, y_train)
R2_test = RSquared(analytic_linreg, X_test, y_test)

In [123]:
df_mae.loc[2] = ['analyticLinearRegression', MAE_train, MAE_test]
df_rmse.loc[2] = ['analyticLinearRegression', RMSE_train, RMSE_test]
df_r2.loc[2] = ['analyticLinearRegression', R2_train, R2_test]

In [124]:
grad_linreg = GradLinearRegression(1000)
grad_linreg.fit(X_train, y_train)

In [125]:
grad_linreg_pred_train = grad_linreg.predict(X_train)
grad_linreg_pred_test = grad_linreg.predict(X_test)

In [126]:
MAE_train = mean_absolute_error(y_train, grad_linreg_pred_train)
MAE_test = mean_absolute_error(y_test, grad_linreg_pred_test)
RMSE_train = np.sqrt(mean_squared_error(y_train, grad_linreg_pred_train))
RMSE_test = np.sqrt(mean_squared_error(y_test, grad_linreg_pred_test))
R2_train = RSquared(grad_linreg, X_train, y_train)
R2_test = RSquared(grad_linreg, X_test, y_test)

In [127]:
df_mae.loc[3] = ['gradLinearRegression', MAE_train, MAE_test]
df_rmse.loc[3] = ['gradLinearRegression', RMSE_train, RMSE_test]
df_r2.loc[3] = ['gradLinearRegression', R2_train, R2_test]

In [128]:
df_mae

,model,train,test
0,SGDLinearRegression,706.020899,707.148104
1,sklearnLinearRegression,708.494998,709.678526
2,analyticLinearRegression,708.494998,709.678526
3,gradLinearRegression,708.698692,710.807768


In [129]:
df_rmse

,model,train,test
0,SGDLinearRegression,1028.526506,1024.031348
1,sklearnLinearRegression,1028.253868,1023.842890
2,analyticLinearRegression,1028.253868,1023.842890
3,gradLinearRegression,1029.572717,1026.391535


In [130]:
df_r2

,model,train,test
0,SGDLinearRegression,0.576490,0.593447
1,sklearnLinearRegression,0.576715,0.593597
2,analyticLinearRegression,0.576715,0.593597
3,gradLinearRegression,0.575628,0.591571


## 5. Regularized models implementation — Ridge, Lasso, ElasticNet

### Store the metrics as in the previous lesson in a table with columns model, train, test for MAE table, RMSE table, and R2 coefficient.

In [131]:
df_mae_reg = pd.DataFrame(columns=['model', 'train', 'test'])
df_rmse_reg = pd.DataFrame(columns=['model', 'train', 'test'])
df_r2_reg = pd.DataFrame(columns=['model', 'train', 'test'])

In [132]:
def PutIn(model, model_name, dataframe1 ,dataframe2, dataframe3):
    model.fit(X_train, y_train)
    model_pred_train = model.predict(X_train)
    model_pred_test = model.predict(X_test)

    MAE_train = mean_absolute_error(y_train, model_pred_train)
    MAE_test = mean_absolute_error(y_test, model_pred_test)
    RMSE_train = np.sqrt(mean_squared_error(y_train, model_pred_train))
    RMSE_test = np.sqrt(mean_squared_error(y_test, model_pred_test))
    R2_train = RSquared(model, X_train, y_train)
    R2_test = RSquared(model, X_test, y_test)

    dataframe1.loc[len(df_mae_reg)] = [model_name, MAE_train, MAE_test]
    dataframe2.loc[len(df_rmse_reg)] = [model_name, RMSE_train, RMSE_test]
    dataframe3.loc[len(df_r2_reg)] = [model_name, R2_train, R2_test]

#### SGDlinreg

In [133]:
MyLassoSGD = SGDLinearRegression(epoch=1000, l1=1)
PutIn(MyLassoSGD, 'LassoSGD', df_mae_reg, df_rmse_reg, df_r2_reg)

In [134]:
MyRidgeSGD = SGDLinearRegression(epoch=1000, l2=1)
PutIn(MyRidgeSGD, 'RidgeSGD', df_mae_reg, df_rmse_reg, df_r2_reg)

In [135]:
MyElasticNetSGD = SGDLinearRegression(epoch=1000, l1=0.5, l2=0.5)
PutIn(MyElasticNetSGD, 'ElasticNetSGD', df_mae_reg, df_rmse_reg, df_r2_reg)

In [136]:
MyLassoGrad = GradLinearRegression(epoch=1000, l1=1)
PutIn(MyLassoGrad, 'LassoGrad', df_mae_reg, df_rmse_reg, df_r2_reg)

In [137]:
MyRidgeGrad = GradLinearRegression(epoch=1000, l2=1, learning_rate=0.001)
PutIn(MyRidgeGrad, 'RidgeGrad', df_mae_reg, df_rmse_reg , df_r2_reg)

In [138]:
MyElasticNetGrad = GradLinearRegression(epoch=1000, l1=0.5, l2=0.5)
PutIn(MyElasticNetGrad, 'ElasticNetGrad', df_mae_reg, df_rmse_reg, df_r2_reg)

In [139]:
MyRidgeAnalytic = AnalLinearRegression(l2=1)
PutIn(MyRidgeAnalytic, 'RidgeAnalytic', df_mae_reg, df_rmse_reg, df_r2_reg)

In [140]:
sklearnLasso = Lasso(alpha=1)
PutIn(sklearnLasso, 'LassoSklearn', df_mae_reg, df_rmse_reg, df_r2_reg)

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Lasso was fitted with feature names
  warnings.warn(
C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Lasso was fitted with feature names
  warnings.warn(


In [141]:
sklearnRidge = Ridge(alpha=1)
PutIn(sklearnRidge, 'RidgeSklearn', df_mae_reg, df_rmse_reg, df_r2_reg)

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(


In [142]:
sklearnElasticNet = ElasticNet(alpha=1.5, l1_ratio=0.3333)
PutIn(sklearnElasticNet, 'ElasticNetSklearn', df_mae_reg, df_rmse_reg, df_r2_reg)

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but ElasticNet was fitted with feature names
  warnings.warn(
C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but ElasticNet was fitted with feature names
  warnings.warn(


In [143]:
df_mae_reg

,model,train,test
0,LassoSGD,706.020880,707.148093
1,RidgeSGD,706.012268,707.143637
2,ElasticNetSGD,706.016559,707.145865
3,LassoGrad,708.673614,710.791665
4,RidgeGrad,775.550767,784.463453
5,ElasticNetGrad,798.702782,805.770275
6,RidgeAnalytic,708.491597,709.677352
7,LassoSklearn,708.093245,709.569265
8,RidgeSklearn,708.491597,709.677352
9,ElasticNetSklearn,864.886275,872.828573


In [144]:
df_rmse_reg

,model,train,test
0,LassoSGD,1028.526506,1024.031353
1,RidgeSGD,1028.526266,1024.037783
2,ElasticNetSGD,1028.526378,1024.034561
3,LassoGrad,1029.751348,1026.626059
4,RidgeGrad,1189.435470,1199.810528
5,ElasticNetGrad,1174.429849,1186.785169
6,RidgeAnalytic,1028.253876,1023.846261
7,LassoSklearn,1028.443194,1024.223384
8,RidgeSklearn,1028.253876,1023.846261
9,ElasticNetSklearn,1256.535525,1272.499946


In [145]:
df_r2_reg

,model,train,test
0,LassoSGD,0.576490,0.593447
1,RidgeSGD,0.576490,0.593442
2,ElasticNetSGD,0.576490,0.593445
3,LassoGrad,0.575481,0.591384
4,RidgeGrad,0.433612,0.441895
5,ElasticNetGrad,0.447812,0.453947
6,RidgeAnalytic,0.576715,0.593594
7,LassoSklearn,0.576559,0.593295
8,RidgeSklearn,0.576715,0.593594
9,ElasticNetSklearn,0.367905,0.372222


## 6. Feature normalization

#### First, write several examples of why and where feature normalization is mandatory and vice versa.

в регресси с регуляризацией обязательно делать нормализацию потому что штраф примнеяется сразу ко всем признакам

нормализация не нужна в дереве принятия решений

#### Let's consider the first of the classical normalization methods — MinMaxScaler. Write a mathematical formula for this method.

minmaxscaler = (X - Xmin) / (Xmax - Xmin)

#### Implement your own function or class for MinMaxScaler feature normalization.


In [146]:
X_train_min = X_train.min()
X_train_max = X_train.max()
def MyMinMaxScaler(X):
    result = (X - X_train_min) / (X_train_max - X_train_min)
    return result

In [147]:
X_train

,bathrooms,bedrooms,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,...,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
110878,1.0,1,1,1,1,1,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
67866,1.0,1,0,0,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
99860,1.0,2,1,0,1,1,1,0,0,0,...,0,0,1,0,0,0,0,0,0,1
9759,2.0,2,1,1,0,0,1,1,1,1,...,0,1,0,1,0,0,1,0,0,1
49188,1.0,2,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42246,1.0,3,1,1,0,0,1,0,1,1,...,1,0,0,0,0,0,0,0,0,0
23135,2.5,4,1,1,0,0,0,1,0,1,...,1,0,0,0,0,0,0,0,0,0
15367,1.0,2,1,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
13753,1.0,1,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [148]:
X_train_scaled_minmax = MyMinMaxScaler(X_train)
X_test_scaled_minmax = MyMinMaxScaler(X_test)
X_train_scaled_minmax

,bathrooms,bedrooms,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,...,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
110878,0.10,0.125,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
67866,0.10,0.125,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
99860,0.10,0.250,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
9759,0.20,0.250,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
49188,0.10,0.250,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42246,0.10,0.375,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
23135,0.25,0.500,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15367,0.10,0.250,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13753,0.10,0.125,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [149]:
minmax = MinMaxScaler()
minmax.fit_transform(X_train)

array([[0.1  , 0.125, 1.   , ..., 0.   , 0.   , 0.   ],
       [0.1  , 0.125, 0.   , ..., 0.   , 0.   , 0.   ],
       [0.1  , 0.25 , 1.   , ..., 0.   , 0.   , 1.   ],
       ...,
       [0.1  , 0.25 , 1.   , ..., 0.   , 0.   , 0.   ],
       [0.1  , 0.125, 0.   , ..., 0.   , 0.   , 0.   ],
       [0.1  , 0.   , 0.   , ..., 0.   , 0.   , 0.   ]], shape=(38674, 22))

they are the same

#### Repeat the steps from b to e for another normalization method StandardScaler.


In [150]:
train_mean = X_train.mean()
train_std = X_train.std(ddof=0)
def MyStandardScaler(X):
    return (X - train_mean) / train_std

In [151]:
X_train_scaled_standard = MyStandardScaler(X_train)
X_test_scaled_standard = MyStandardScaler(X_test)
X_train_scaled_standard

,bathrooms,bedrooms,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,...,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
110878,-0.427098,-0.486642,0.950936,1.041400,1.046213,1.112341,-0.857344,1.185117,-0.760993,1.413994,...,-0.459165,-0.391243,-0.344598,-0.335369,-0.309212,-0.252967,-0.240042,-0.233772,-0.234755,-0.218159
67866,-0.427098,-0.486642,-1.051596,-0.960246,1.046213,1.112341,1.166393,-0.843799,-0.760993,-0.707216,...,-0.459165,-0.391243,-0.344598,-0.335369,-0.309212,-0.252967,-0.240042,-0.233772,-0.234755,-0.218159
99860,-0.427098,0.424361,0.950936,-0.960246,1.046213,1.112341,1.166393,-0.843799,-0.760993,-0.707216,...,-0.459165,-0.391243,2.901929,-0.335369,-0.309212,-0.252967,-0.240042,-0.233772,-0.234755,4.583818
9759,1.774658,0.424361,0.950936,1.041400,-0.955828,-0.899005,1.166393,1.185117,1.314072,1.413994,...,-0.459165,2.555957,-0.344598,2.981786,-0.309212,-0.252967,4.165934,-0.233772,-0.234755,4.583818
49188,-0.427098,0.424361,0.950936,-0.960246,-0.955828,-0.899005,-0.857344,-0.843799,-0.760993,-0.707216,...,-0.459165,-0.391243,-0.344598,-0.335369,-0.309212,-0.252967,-0.240042,-0.233772,-0.234755,-0.218159
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42246,-0.427098,1.335364,0.950936,1.041400,-0.955828,-0.899005,1.166393,-0.843799,1.314072,1.413994,...,2.177865,-0.391243,-0.344598,-0.335369,-0.309212,-0.252967,-0.240042,-0.233772,-0.234755,-0.218159
23135,2.875536,2.246367,0.950936,1.041400,-0.955828,-0.899005,-0.857344,1.185117,-0.760993,1.413994,...,2.177865,-0.391243,-0.344598,-0.335369,-0.309212,-0.252967,-0.240042,-0.233772,-0.234755,-0.218159
15367,-0.427098,0.424361,0.950936,1.041400,-0.955828,-0.899005,1.166393,-0.843799,-0.760993,-0.707216,...,-0.459165,-0.391243,-0.344598,-0.335369,-0.309212,-0.252967,-0.240042,-0.233772,-0.234755,-0.218159
13753,-0.427098,-0.486642,-1.051596,-0.960246,1.046213,1.112341,-0.857344,-0.843799,-0.760993,-0.707216,...,-0.459165,-0.391243,-0.344598,-0.335369,-0.309212,-0.252967,-0.240042,-0.233772,-0.234755,-0.218159


In [152]:
stdScaler = StandardScaler()
stdScaler.fit_transform(X_train)

array([[-0.42709754, -0.48664247,  0.9509358 , ..., -0.23377249,
        -0.23475508, -0.21815877],
       [-0.42709754, -0.48664247, -1.0515957 , ..., -0.23377249,
        -0.23475508, -0.21815877],
       [-0.42709754,  0.42436052,  0.9509358 , ..., -0.23377249,
        -0.23475508,  4.58381752],
       ...,
       [-0.42709754,  0.42436052,  0.9509358 , ..., -0.23377249,
        -0.23475508, -0.21815877],
       [-0.42709754, -0.48664247, -1.0515957 , ..., -0.23377249,
        -0.23475508, -0.21815877],
       [-0.42709754, -1.39764546, -1.0515957 , ..., -0.23377249,
        -0.23475508, -0.21815877]], shape=(38674, 22))

they are the same

## 7. Fit custom and sklearn models with normalized data



#### Fit all models — Linear Regression, Ridge, Lasso, and ElasticNet — with MinMaxScaler.

In [153]:
df_mae_minmax = pd.DataFrame(columns=['model', 'train', 'test'])
df_rmse_minmax = pd.DataFrame(columns=['model', 'train', 'test'])
df_r2_minmax = pd.DataFrame(columns=['model', 'train', 'test'])

In [154]:
def PutInMinMax(model, model_name, dataframe1 ,dataframe2, dataframe3):
    model.fit(X_train_scaled_minmax, y_train)
    model_pred_train = model.predict(X_train_scaled_minmax)
    model_pred_test = model.predict(X_test_scaled_minmax)

    MAE_train = mean_absolute_error(y_train, model_pred_train)
    MAE_test = mean_absolute_error(y_test, model_pred_test)
    RMSE_train = np.sqrt(mean_squared_error(y_train, model_pred_train))
    RMSE_test = np.sqrt(mean_squared_error(y_test, model_pred_test))
    R2_train = RSquared(model, X_train_scaled_minmax, y_train)
    R2_test = RSquared(model, X_test_scaled_minmax, y_test)

    dataframe1.loc[len(dataframe1)] = [model_name, MAE_train, MAE_test]
    dataframe2.loc[len(dataframe2)] = [model_name, RMSE_train, RMSE_test]
    dataframe3.loc[len(dataframe3)] = [model_name, R2_train, R2_test]

In [155]:
sgd_linreg_minmax = SGDLinearRegression(epoch=1000)
PutInMinMax(sgd_linreg_minmax, 'SGDLinearRegression', df_mae_minmax, df_rmse_minmax, df_r2_minmax)

In [156]:
sgd_lasso_minmax = SGDLinearRegression(epoch=1000, l1=1)
PutInMinMax(sgd_lasso_minmax, 'SGDLasso', df_mae_minmax, df_rmse_minmax, df_r2_minmax)

In [157]:
sgd_ridge_minmax = SGDLinearRegression(epoch=1000, l2=1)
PutInMinMax(sgd_ridge_minmax, 'SGDRidge', df_mae_minmax, df_rmse_minmax, df_r2_minmax)

In [158]:
sgd_elasticnet_minmax = SGDLinearRegression(epoch=1000, l1=0.5, l2=0.5)
PutInMinMax(sgd_elasticnet_minmax, 'SGDElasticNet', df_mae_minmax, df_rmse_minmax, df_r2_minmax)

In [159]:
grad_linreg_minmax = GradLinearRegression(epoch=1000)
PutInMinMax(grad_linreg_minmax, 'GradLinearRegression', df_mae_minmax, df_rmse_minmax, df_r2_minmax)

In [160]:
grad_lasso_minmax = GradLinearRegression(epoch=1000, l1=1)
PutInMinMax(grad_lasso_minmax, 'GradLasso', df_mae_minmax, df_rmse_minmax, df_r2_minmax)

In [161]:
grad_ridge_minmax = GradLinearRegression(epoch=1000, l2=2)
PutInMinMax(grad_ridge_minmax, 'GradRidge', df_mae_minmax, df_rmse_minmax, df_r2_minmax)

In [162]:
grad_elasticnet_minmax = GradLinearRegression(epoch=1000, l1=0.5, l2=0.5)
PutInMinMax(grad_elasticnet_minmax, 'GradElasticNet', df_mae_minmax, df_rmse_minmax, df_r2_minmax)

In [163]:
analytic_ridge = AnalLinearRegression(l2=1)
PutInMinMax(analytic_ridge, 'AnalyticRidge', df_mae_minmax, df_rmse_minmax, df_r2_minmax)

In [164]:
sklearn_Linreg_minmax = LinearRegression()
PutInMinMax(sklearn_Linreg_minmax, 'sklearnLinreg', df_mae_minmax, df_rmse_minmax, df_r2_minmax)

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [165]:
sklearn_lasso_minmax = Lasso(alpha=1)
PutInMinMax(sklearn_lasso_minmax, 'sklearnLasso', df_mae_minmax, df_rmse_minmax, df_r2_minmax)

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Lasso was fitted with feature names
  warnings.warn(
C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Lasso was fitted with feature names
  warnings.warn(


In [166]:
sklearn_ridge_minmax = Ridge(alpha=1)
PutInMinMax(sklearn_ridge_minmax, 'sklearnRidge', df_mae_minmax, df_rmse_minmax, df_r2_minmax)

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(


In [167]:
sklearn_elasticnet_minmax = ElasticNet(alpha=1.5, l1_ratio=0.3333)
PutInMinMax(sklearn_elasticnet_minmax, 'sklearnElasticNet', df_mae_minmax, df_rmse_minmax, df_r2_minmax)

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but ElasticNet was fitted with feature names
  warnings.warn(
C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but ElasticNet was fitted with feature names
  warnings.warn(


In [168]:
df_mae_minmax

,model,train,test
0,SGDLinearRegression,707.473503,708.629463
1,SGDLasso,707.473491,708.629458
2,SGDRidge,707.761065,709.064464
3,SGDElasticNet,707.594431,708.829570
4,GradLinearRegression,884.626237,893.047662
5,GradLasso,885.202328,893.707055
6,GradRidge,1093.946201,1106.870694
7,GradElasticNet,1050.380648,1062.842981
8,AnalyticRidge,708.614152,709.884872
9,sklearnLinreg,708.494998,709.678526


In [169]:
df_rmse_minmax

,model,train,test
0,SGDLinearRegression,1028.318320,1023.880161
1,SGDLasso,1028.318319,1023.880173
2,SGDRidge,1028.499598,1024.408752
3,SGDElasticNet,1028.362925,1024.100186
4,GradLinearRegression,1277.061106,1288.410989
5,GradLasso,1277.775524,1289.263619
6,GradRidge,1531.489596,1555.807299
7,GradElasticNet,1476.840704,1498.739476
8,AnalyticRidge,1028.303518,1024.070144
9,sklearnLinreg,1028.253868,1023.842890


In [170]:
df_r2_minmax

,model,train,test
0,SGDLinearRegression,0.576662,0.593567
1,SGDLasso,0.576662,0.593567
2,SGDRidge,0.576512,0.593147
3,SGDElasticNet,0.576625,0.593392
4,GradLinearRegression,0.347086,0.356425
5,GradLasso,0.346355,0.355573
6,GradRidge,0.061011,0.061569
7,GradElasticNet,0.126828,0.129151
8,AnalyticRidge,0.576674,0.593416
9,sklearnLinreg,0.576715,0.593597


#### Fit all models — Linear Regression, Ridge, Lasso, and ElasticNet — with StandardScaler.


In [171]:
df_mae_std_scaler = pd.DataFrame(columns=['model', 'train', 'test'])
df_rmse_std_scaler = pd.DataFrame(columns=['model', 'train', 'test'])
df_r2_std_scaler = pd.DataFrame(columns=['model', 'train', 'test'])

In [172]:
def PutInStdScaler(model, model_name):
    model.fit(X_train_scaled_standard, y_train)
    model_pred_train = model.predict(X_train_scaled_standard)
    model_pred_test = model.predict(X_test_scaled_standard)

    MAE_train = mean_absolute_error(y_train, model_pred_train)
    MAE_test = mean_absolute_error(y_test, model_pred_test)
    RMSE_train = np.sqrt(mean_squared_error(y_train, model_pred_train))
    RMSE_test = np.sqrt(mean_squared_error(y_test, model_pred_test))
    R2_train = RSquared(model, X_train_scaled_standard, y_train)
    R2_test = RSquared(model, X_test_scaled_standard, y_test)

    df_mae_std_scaler.loc[len(df_mae_std_scaler)] = [model_name, MAE_train, MAE_test]
    df_rmse_std_scaler.loc[len(df_rmse_std_scaler)] = [model_name, RMSE_train, RMSE_test]
    df_r2_std_scaler.loc[len(df_r2_std_scaler)] = [model_name, R2_train, R2_test]

In [173]:
sgd_linreg_std = SGDLinearRegression(epoch=1000)
PutInStdScaler(sgd_linreg_std, 'SGDLinearRegression')

In [174]:
sgd_lasso_std = SGDLinearRegression(epoch=1000, l1=1)
PutInStdScaler(sgd_lasso_std, 'SGDLasso')

In [175]:
sgd_ridge_std = SGDLinearRegression(epoch=1000, l2=1)
PutInStdScaler(sgd_ridge_std, 'SGDRidge')

In [176]:
sgd_elasticNet_std = SGDLinearRegression(epoch=1000, l1=0.5, l2=0.5)
PutInStdScaler(sgd_elasticNet_std, 'SGDElasticNet')

In [177]:
grad_linreg_std = GradLinearRegression(epoch=1000)
PutInStdScaler(grad_linreg_std, 'GradLinearRegression')

In [178]:
grad_lasso_std = GradLinearRegression(epoch=1000, l1=1)
PutInStdScaler(grad_lasso_std, 'GradLasso')

In [179]:
grad_ridge_std = GradLinearRegression(epoch=1000, l2=1)
PutInStdScaler(grad_ridge_std, 'GradRidge')

In [180]:
grad_elasticnet_std = GradLinearRegression(epoch=1000, l1=0.5, l2=0.5)
PutInStdScaler(grad_elasticnet_std, 'GradElasticNet')

In [181]:
my_analytic = AnalLinearRegression()
PutInStdScaler(my_analytic, 'AnalyticLinearRegression')

In [182]:
analytic_ridge_std = AnalLinearRegression(l2=1)
PutInStdScaler(analytic_ridge_std, 'AnalyticRidge')

In [183]:
sklearn_Linreg_std = LinearRegression()
PutInStdScaler(sklearn_Linreg_std, 'SklearnLinearRegression')

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [184]:
sklearn_lasso_std = Lasso(alpha=1)
PutInStdScaler(sklearn_lasso_std, 'SklearnLasso')

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Lasso was fitted with feature names
  warnings.warn(
C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Lasso was fitted with feature names
  warnings.warn(


In [185]:
sklearn_ridge_std = Ridge(alpha=1)
PutInStdScaler(sklearn_ridge_std, 'SklearnRidge')

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(


In [186]:
sklearn_elastic_net_std = ElasticNet(alpha=1.5, l1_ratio=0.3333)
PutInStdScaler(sklearn_elastic_net_std, 'SklearnElasticNet')

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but ElasticNet was fitted with feature names
  warnings.warn(
C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but ElasticNet was fitted with feature names
  warnings.warn(


In [187]:
df_mae_std_scaler

,model,train,test
0,SGDLinearRegression,709.551396,710.482229
1,SGDLasso,709.551389,710.482225
2,SGDRidge,709.549504,710.480416
3,SGDElasticNet,709.550446,710.481321
4,GradLinearRegression,708.520071,709.778514
5,GradLasso,708.423631,709.713733
6,GradRidge,784.525364,789.581031
7,GradElasticNet,739.131195,742.498785
8,AnalyticLinearRegression,708.494998,709.678526
9,AnalyticRidge,708.494411,709.677980


In [188]:
df_rmse_std_scaler

,model,train,test
0,SGDLinearRegression,1029.101612,1024.150493
1,SGDLasso,1029.101612,1024.150495
2,SGDRidge,1029.100826,1024.150512
3,SGDElasticNet,1029.101219,1024.150503
4,GradLinearRegression,1028.265559,1023.917263
5,GradLasso,1028.277299,1023.953985
6,GradRidge,1131.338810,1137.099203
7,GradElasticNet,1072.556160,1074.115153
8,AnalyticLinearRegression,1028.253868,1023.842890
9,AnalyticRidge,1028.253868,1023.843308


In [189]:
df_r2_std_scaler

,model,train,test
0,SGDLinearRegression,0.576016,0.593353
1,SGDLasso,0.576016,0.593353
2,SGDRidge,0.576017,0.593353
3,SGDElasticNet,0.576017,0.593353
4,GradLinearRegression,0.576705,0.593538
5,GradLasso,0.576695,0.593509
6,GradRidge,0.487590,0.498712
7,GradElasticNet,0.539454,0.552707
8,AnalyticLinearRegression,0.576715,0.593597
9,AnalyticRidge,0.576715,0.593596


## 8. Overfit models

#### In the previous lesson, we created polynomial features with degree 10. Here we repeat these steps from the previous lesson, remembering that we have only 2 basic features — 'bathrooms' and 'bedrooms'.

In [190]:
numeric_features_train = X_train[['bathrooms', 'bedrooms']]
numeric_features_test = X_test[['bathrooms', 'bedrooms']]
poly = PolynomialFeatures(degree=2, include_bias=False)
numeric_train_poly = poly.fit_transform(numeric_features_train)
numeric_test_poly = poly.transform(numeric_features_test)
new_cols = poly.get_feature_names_out(numeric_features_train.columns)
numeric_train_poly_df = pd.DataFrame(numeric_train_poly, columns=new_cols, index=numeric_features_train.index)
numeric_test_poly_df = pd.DataFrame(numeric_test_poly, columns=new_cols, index=numeric_features_test.index)
numeric_train_poly_df

,bathrooms,bedrooms,bathrooms^2,bathrooms bedrooms,bedrooms^2
110878,1.0,1.0,1.00,1.0,1.0
67866,1.0,1.0,1.00,1.0,1.0
99860,1.0,2.0,1.00,2.0,4.0
9759,2.0,2.0,4.00,4.0,4.0
49188,1.0,2.0,1.00,2.0,4.0
...,...,...,...,...,...
42246,1.0,3.0,1.00,3.0,9.0
23135,2.5,4.0,6.25,10.0,16.0
15367,1.0,2.0,1.00,2.0,4.0
13753,1.0,1.0,1.00,1.0,1.0


In [191]:
X_train_poly = X_train.drop(columns=['bathrooms', 'bedrooms'])
X_test_poly = X_test.drop(columns=['bathrooms', 'bedrooms'])
X_train_poly = pd.concat([numeric_train_poly_df, X_train_poly], axis=1)
X_test_poly = pd.concat([numeric_test_poly_df, X_test_poly], axis=1)
X_train_poly

,bathrooms,bedrooms,bathrooms^2,bathrooms bedrooms,bedrooms^2,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,...,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
110878,1.0,1.0,1.00,1.0,1.0,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
67866,1.0,1.0,1.00,1.0,1.0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
99860,1.0,2.0,1.00,2.0,4.0,1,0,1,1,1,...,0,0,1,0,0,0,0,0,0,1
9759,2.0,2.0,4.00,4.0,4.0,1,1,0,0,1,...,0,1,0,1,0,0,1,0,0,1
49188,1.0,2.0,1.00,2.0,4.0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42246,1.0,3.0,1.00,3.0,9.0,1,1,0,0,1,...,1,0,0,0,0,0,0,0,0,0
23135,2.5,4.0,6.25,10.0,16.0,1,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
15367,1.0,2.0,1.00,2.0,4.0,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
13753,1.0,1.0,1.00,1.0,1.0,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0


#### And train and fit all our implemented algorithms — Linear Regression, Ridge, Lasso, and ElasticNet — on a set of polynomial features.

In [192]:
df_mae_poly = pd.DataFrame(columns=['model', 'train', 'test'])
df_rmse_poly = pd.DataFrame(columns=['model', 'train', 'test'])
df_r2_poly = pd.DataFrame(columns=['model', 'train', 'test'])

In [193]:
def PutInPoly(model, model_name):
    model.fit(X_train_poly, y_train)
    model_pred_train = model.predict(X_train_poly)
    model_pred_test = model.predict(X_test_poly)

    MAE_train = mean_absolute_error(y_train, model_pred_train)
    MAE_test = mean_absolute_error(y_test, model_pred_test)
    RMSE_train = np.sqrt(mean_squared_error(y_train, model_pred_train))
    RMSE_test = np.sqrt(mean_squared_error(y_test, model_pred_test))
    R2_train = RSquared(model, X_train_poly, y_train)
    R2_test = RSquared(model, X_test_poly, y_test)

    df_mae_poly.loc[len(df_mae_poly)] = [model_name, MAE_train, MAE_test]
    df_rmse_poly.loc[len(df_rmse_poly)] = [model_name, RMSE_train, RMSE_test]
    df_r2_poly.loc[len(df_r2_poly)] = [model_name, R2_train, R2_test]

In [194]:
sgd_linreg_poly = SGDLinearRegression(epoch=1000)
PutInPoly(sgd_linreg_poly, 'SGDLinearRegression')

In [195]:
sgd_lasso_poly = SGDLinearRegression(epoch=1000, l1=1)
PutInPoly(sgd_lasso_poly, 'SGDLasso')

In [196]:
sgd_ridge_poly = SGDLinearRegression(epoch=1000, l2=1)
PutInPoly(sgd_ridge_poly, 'SGDRidge')

In [197]:
sgd_elasticnet_poly = SGDLinearRegression(epoch=1000, l1=0.5, l2=0.5)
PutInPoly(sgd_elasticnet_poly, 'SGDElasticNet')

In [198]:
grad_linreg_poly = GradLinearRegression(epoch=1000)
PutInPoly(grad_linreg_poly, 'GradLinearRegression')

In [199]:
grad_lasso_poly = GradLinearRegression(epoch=1000, l1=1)
PutInPoly(grad_lasso_poly, 'GradLasso')

In [200]:
grad_rigde_poly = GradLinearRegression(epoch=1000, l2=1)
PutInPoly(grad_rigde_poly, 'GradRidge')

In [201]:
grad_elasticnet_poly = GradLinearRegression(epoch=1000, l1=0.5, l2=0.5)
PutInPoly(grad_elasticnet_poly, 'GradElasticNet')

In [202]:
analytic_linreg_poly = AnalLinearRegression()
PutInPoly(analytic_linreg_poly, 'AnalyticLinearRegression')

In [203]:
analytic_ridge_poly = AnalLinearRegression(l2=1)
PutInPoly(analytic_ridge_poly, 'AnalyticRidge')

In [204]:
sklearn_linreg_poly = LinearRegression()
PutInPoly(sklearn_linreg_poly, 'sklearnLinearRegression')

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [205]:
sklearn_lasso_poly = Lasso(alpha=0.2)
PutInPoly(sklearn_lasso_poly, 'SklearnLasso')

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Lasso was fitted with feature names
  warnings.warn(
C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Lasso was fitted with feature names
  warnings.warn(


In [206]:
sklearn_ridge_poly = Ridge(alpha=1)
PutInPoly(sklearn_ridge_poly, 'SklearnRidge')

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(


In [207]:
sklearn_elasticnet_poly = ElasticNet(alpha=1.5, l1_ratio=0.333)
PutInPoly(sklearn_elasticnet_poly, 'SklearnElasticNet')

C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but ElasticNet was fitted with feature names
  warnings.warn(
C:\Users\130\school_21_ds_1\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but ElasticNet was fitted with feature names
  warnings.warn(


In [208]:
df_mae_poly

,model,train,test
0,SGDLinearRegression,701.053203,701.464906
1,SGDLasso,701.053177,701.464884
2,SGDRidge,701.031634,701.448832
3,SGDElasticNet,701.042302,701.456849
4,GradLinearRegression,703.968297,707.027794
5,GradLasso,703.900434,706.948924
6,GradRidge,754.277644,761.410170
7,GradElasticNet,734.111557,740.997491
8,AnalyticLinearRegression,706.054383,707.959202
9,AnalyticRidge,706.044479,707.952818


In [209]:
df_rmse_poly

,model,train,test
0,SGDLinearRegression,1046.192685,1040.732733
1,SGDLasso,1046.192694,1040.732742
2,SGDRidge,1046.210291,1040.749831
3,SGDElasticNet,1046.201474,1040.741266
4,GradLinearRegression,1028.466969,1021.392820
5,GradLasso,1028.635678,1021.545245
6,GradRidge,1099.450412,1103.246279
7,GradElasticNet,1076.522321,1078.125997
8,AnalyticLinearRegression,1026.280363,1021.224109
9,AnalyticRidge,1026.280393,1021.226391


In [210]:
df_r2_poly

,model,train,test
0,SGDLinearRegression,0.561817,0.580078
1,SGDLasso,0.561817,0.580078
2,SGDRidge,0.561802,0.580064
3,SGDElasticNet,0.561809,0.580071
4,GradLinearRegression,0.576539,0.595539
5,GradLasso,0.576400,0.595419
6,GradRidge,0.516068,0.528116
7,GradElasticNet,0.536042,0.549360
8,AnalyticLinearRegression,0.578338,0.595673
9,AnalyticRidge,0.578338,0.595671


## 9. Native models



In [211]:
price_mean_train = np.full_like(y_train, y_train.mean())
price_mean_test = np.full_like(y_test, y_test.mean())
price_median_train = np.full_like(y_train, y_train.median())
price_median_test = np.full_like(y_test, y_test.median())

In [212]:
MAE_mean_train = mean_absolute_error(y_train, price_mean_train)
MAE_mean_test = mean_absolute_error(y_test, price_mean_test)
MAE_median_train = mean_absolute_error(y_train, price_median_train)
MAE_median_test = mean_absolute_error(y_test, price_median_test)
RMSE_mean_train = np.sqrt(mean_squared_error(y_train, price_mean_train))
RMSE_mean_test = np.sqrt(mean_squared_error(y_test, price_mean_test))
RMSE_median_train = np.sqrt(mean_squared_error(y_train, price_median_train))
RMSE_median_test = np.sqrt(mean_squared_error(y_test, price_median_test))
R2_mean_train = 0
R2_mean_test = 0
R2_median_train = 1 - (np.sum((y_train - price_median_train)**2)) / np.sum((y_train - price_mean_train)**2)
R2_median_test = 1 - (np.sum((y_test - price_median_test)**2)) / np.sum((y_test - price_mean_test)**2)

In [213]:
df_mae_poly.loc[len(df_mae_poly)] = ['naive_mean', MAE_mean_train, MAE_mean_test]
df_mae_poly.loc[len(df_mae_poly)] = ['naive_median', MAE_median_train, MAE_median_test]
df_rmse_poly.loc[len(df_rmse_poly)] = ['naive_mean', RMSE_mean_train, RMSE_mean_test]
df_rmse_poly.loc[len(df_rmse_poly)] = ['naive_median', RMSE_median_train, RMSE_median_test]
df_r2_poly.loc[len(df_r2_poly)] = ['naive_mean', R2_mean_train, R2_mean_test]
df_r2_poly.loc[len(df_r2_poly)] = ['naive_median', R2_median_train, R2_median_test]

## 10. Compare results



In [214]:
df_mae_poly

,model,train,test
0,SGDLinearRegression,701.053203,701.464906
1,SGDLasso,701.053177,701.464884
2,SGDRidge,701.031634,701.448832
3,SGDElasticNet,701.042302,701.456849
4,GradLinearRegression,703.968297,707.027794
5,GradLasso,703.900434,706.948924
6,GradRidge,754.277644,761.410170
7,GradElasticNet,734.111557,740.997491
8,AnalyticLinearRegression,706.054383,707.959202
9,AnalyticRidge,706.044479,707.952818


In [215]:
df_rmse_poly

,model,train,test
0,SGDLinearRegression,1046.192685,1040.732733
1,SGDLasso,1046.192694,1040.732742
2,SGDRidge,1046.210291,1040.749831
3,SGDElasticNet,1046.201474,1040.741266
4,GradLinearRegression,1028.466969,1021.392820
5,GradLasso,1028.635678,1021.545245
6,GradRidge,1099.450412,1103.246279
7,GradElasticNet,1076.522321,1078.125997
8,AnalyticLinearRegression,1026.280363,1021.224109
9,AnalyticRidge,1026.280393,1021.226391


In [216]:
df_r2_poly

,model,train,test
0,SGDLinearRegression,0.561817,0.580078
1,SGDLasso,0.561817,0.580078
2,SGDRidge,0.561802,0.580064
3,SGDElasticNet,0.561809,0.580071
4,GradLinearRegression,0.576539,0.595539
5,GradLasso,0.576400,0.595419
6,GradRidge,0.516068,0.528116
7,GradElasticNet,0.536042,0.549360
8,AnalyticLinearRegression,0.578338,0.595673
9,AnalyticRidge,0.578338,0.595671


In [217]:
df_mae_minmax

,model,train,test
0,SGDLinearRegression,707.473503,708.629463
1,SGDLasso,707.473491,708.629458
2,SGDRidge,707.761065,709.064464
3,SGDElasticNet,707.594431,708.829570
4,GradLinearRegression,884.626237,893.047662
5,GradLasso,885.202328,893.707055
6,GradRidge,1093.946201,1106.870694
7,GradElasticNet,1050.380648,1062.842981
8,AnalyticRidge,708.614152,709.884872
9,sklearnLinreg,708.494998,709.678526


In [218]:
df_rmse_minmax

,model,train,test
0,SGDLinearRegression,1028.318320,1023.880161
1,SGDLasso,1028.318319,1023.880173
2,SGDRidge,1028.499598,1024.408752
3,SGDElasticNet,1028.362925,1024.100186
4,GradLinearRegression,1277.061106,1288.410989
5,GradLasso,1277.775524,1289.263619
6,GradRidge,1531.489596,1555.807299
7,GradElasticNet,1476.840704,1498.739476
8,AnalyticRidge,1028.303518,1024.070144
9,sklearnLinreg,1028.253868,1023.842890


In [219]:
df_r2_minmax

,model,train,test
0,SGDLinearRegression,0.576662,0.593567
1,SGDLasso,0.576662,0.593567
2,SGDRidge,0.576512,0.593147
3,SGDElasticNet,0.576625,0.593392
4,GradLinearRegression,0.347086,0.356425
5,GradLasso,0.346355,0.355573
6,GradRidge,0.061011,0.061569
7,GradElasticNet,0.126828,0.129151
8,AnalyticRidge,0.576674,0.593416
9,sklearnLinreg,0.576715,0.593597


In [220]:
df_mae_std_scaler

,model,train,test
0,SGDLinearRegression,709.551396,710.482229
1,SGDLasso,709.551389,710.482225
2,SGDRidge,709.549504,710.480416
3,SGDElasticNet,709.550446,710.481321
4,GradLinearRegression,708.520071,709.778514
5,GradLasso,708.423631,709.713733
6,GradRidge,784.525364,789.581031
7,GradElasticNet,739.131195,742.498785
8,AnalyticLinearRegression,708.494998,709.678526
9,AnalyticRidge,708.494411,709.677980


In [221]:
df_rmse_std_scaler

,model,train,test
0,SGDLinearRegression,1029.101612,1024.150493
1,SGDLasso,1029.101612,1024.150495
2,SGDRidge,1029.100826,1024.150512
3,SGDElasticNet,1029.101219,1024.150503
4,GradLinearRegression,1028.265559,1023.917263
5,GradLasso,1028.277299,1023.953985
6,GradRidge,1131.338810,1137.099203
7,GradElasticNet,1072.556160,1074.115153
8,AnalyticLinearRegression,1028.253868,1023.842890
9,AnalyticRidge,1028.253868,1023.843308


In [222]:
df_r2_std_scaler

,model,train,test
0,SGDLinearRegression,0.576016,0.593353
1,SGDLasso,0.576016,0.593353
2,SGDRidge,0.576017,0.593353
3,SGDElasticNet,0.576017,0.593353
4,GradLinearRegression,0.576705,0.593538
5,GradLasso,0.576695,0.593509
6,GradRidge,0.487590,0.498712
7,GradElasticNet,0.539454,0.552707
8,AnalyticLinearRegression,0.576715,0.593597
9,AnalyticRidge,0.576715,0.593596


#### SGDRidge is the best model and Analytic solution is most stable

## Addition task

There are some tricks with the target variable for better model quality. If we have a distribution with a heavy tail, you can use a monotone function to "improve" the distribution. In practice, you can use logarithmic functions. We recommend that you do this exercise and compare the results. But don't forget to do the inverse transformation if you want to compare metrics.


In [223]:
X_train['bedrooms']

110878    1
67866     1
99860     2
9759      2
49188     2
         ..
42246     3
23135     4
15367     2
13753     1
39273     0
Name: bedrooms, Length: 38674, dtype: int64

In [224]:
iqr_bedrooms_train = X_train['bedrooms'].quantile(0.75) - X_train['bedrooms'].quantile(0.25)
lower = X_train['bedrooms'].quantile(0.25) - 1.5 * iqr_bedrooms_train
higher = X_train['bedrooms'].quantile(0.75) + 1.5 * iqr_bedrooms_train
mask = (X_train['bedrooms'] < higher) & (X_train['bedrooms'] > lower)
X_train = X_train[mask]
y_train = y_train[mask]
X_train['bedrooms']

110878    1
67866     1
99860     2
9759      2
49188     2
         ..
108078    1
42246     3
15367     2
13753     1
39273     0
Name: bedrooms, Length: 37078, dtype: int64

X_train['bathrooms']


In [226]:
def RSquaredLn(model, X, Y):
    X = np.array(X)
    y = np.array(Y)
    predict = model.predict(X)
    predict = np.exp(predict)
    MSE = np.sum((y - predict) ** 2)
    naive = np.sum((y - np.mean(y)) ** 2)
    return 1 - MSE/naive

In [227]:
y_train_ln = np.log(y_train)
sgd_ridge_ln = SGDLinearRegression(epoch=1500, l2=1 )
sgd_ridge_ln.fit(X_train, y_train_ln)

In [228]:
predict_ln_train = sgd_ridge_ln.predict(X_train)
predict_train = np.exp(predict_ln_train)
predict_ln_test = sgd_ridge_ln.predict(X_test)
predict_test = np.exp(predict_ln_test)

In [229]:
df_mae_ln = pd.DataFrame(columns=['model_name', 'train', 'test'])
df_rmse_ln = pd.DataFrame(columns=['model_name', 'train', 'test'])
df_r2_ln = pd.DataFrame(columns=['model_name', 'train', 'test'])

In [230]:
MAE_train = mean_absolute_error(y_train, predict_train)
MAE_test = mean_absolute_error(y_test, predict_test)
RMSE_train = np.sqrt(mean_squared_error(y_train, predict_train))
RMSE_test = np.sqrt(mean_squared_error(y_test, predict_test))
R2_train = RSquaredLn(sgd_ridge_ln, X_train, y_train)
R2_test = RSquaredLn(sgd_ridge_ln, X_test, y_test)

In [231]:
df_mae_ln.loc[len(df_mae_ln)] = ['ridge', MAE_train, MAE_test]
df_rmse_ln.loc[len(df_rmse_ln)] = ['ridge', RMSE_train, RMSE_test]
df_r2_ln.loc[len(df_r2_ln)] = ['ridge', R2_train, R2_test]

In [232]:
df_mae_ln

,model_name,train,test
0,ridge,672.07845,697.790125


In [233]:
df_rmse_ln

,model_name,train,test
0,ridge,1033.301944,1035.589971


In [234]:
df_r2_ln

,model_name,train,test
0,ridge,0.50979,0.584218
